# Citation Accuracy Evaluation

This notebook provides a framework for manually annotating and evaluating citation accuracy of agent responses.

**Workflow:**
1. Load qualitative and quantitative KPI queries
2. Create annotation dataframe with columns for manual evaluation
3. Export to CSV for annotation
4. Reload annotated data for analysis

In [12]:
import pandas as pd
import numpy as np
import os
import sys
from dataclasses import fields as dataclass_fields
from typing import List, Tuple, Any

# Add parent directory to path for imports
sys.path.insert(0, os.path.abspath('..'))

from agent.query_quantitative_kpi import QUANTITATIVE_KPIs_speedboat
from agent.query_qualtitative_kpi import QualitativeKPIs_speedboat

## 1. Configuration

In [ ]:
# Configuration
CLIENT = ""  # Change to  client as needed
YEAR_BASIS = "fiscal year"
YEAR_0 = 2024  # Most recent year
YEAR_1 = 2023
YEAR_2 = 2022

# Output paths
OUTPUT_DIR = "citation_evaluation_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Load Qualitative KPI Queries

In [ ]:
# Build qualitative queries
qualitative_kpis = QualitativeKPIs_speedboat(client=CLIENT)
qualitative_kpis.build_queries()

# Extract KPI names and queries
qualitative_data = []
for field in dataclass_fields(qualitative_kpis):
    if field.name in ("client", "_built"):
        continue
    query = getattr(qualitative_kpis, field.name)
    if query is not None:
        qualitative_data.append({
            "kpi_name": field.name,
            "kpi_type": "qualitative",
            "query": query,
            "year": None,
            "client": CLIENT
        })

df_qualitative = pd.DataFrame(qualitative_data)
print(f"Loaded {len(df_qualitative)} qualitative KPIs")
df_qualitative.head()

## 3. Load Quantitative KPI Queries

In [ ]:
# Build quantitative queries
quantitative_kpis = QUANTITATIVE_KPIs_speedboat(client=CLIENT)
quantitative_kpis.build_queries(
    year_basis=YEAR_BASIS,
    year_0=YEAR_0,
    year_1=YEAR_1,
    year_2=YEAR_2
)

# Extract KPI names and queries with year info
quantitative_data = []
for field in dataclass_fields(quantitative_kpis):
    if field.name in ("client", "_built"):
        continue
    query = getattr(quantitative_kpis, field.name)
    if query is not None:
        # Extract year suffix (0, 1, 2) and map to actual year
        kpi_base_name = field.name
        year = None
        if field.name.endswith("_0"):
            kpi_base_name = field.name[:-2]
            year = YEAR_0
        elif field.name.endswith("_1"):
            kpi_base_name = field.name[:-2]
            year = YEAR_1
        elif field.name.endswith("_2"):
            kpi_base_name = field.name[:-2]
            year = YEAR_2
            
        quantitative_data.append({
            "kpi_name": kpi_base_name,
            "kpi_name_full": field.name,
            "kpi_type": "quantitative",
            "query": query,
            "year": year,
            "client": CLIENT
        })

df_quantitative = pd.DataFrame(quantitative_data)
print(f"Loaded {len(df_quantitative)} quantitative KPIs")
df_quantitative.head()

## 4. Create Combined Annotation DataFrame

This dataframe includes columns for manual annotation of citation accuracy.

In [ ]:
# Combine qualitative and quantitative
df_qualitative["kpi_name_full"] = df_qualitative["kpi_name"]  # Add for consistency
df_all = pd.concat([df_qualitative, df_quantitative], ignore_index=True)

# Reorder columns
df_all = df_all[["client", "kpi_type", "kpi_name", "kpi_name_full", "year"]]

print(f"Total KPIs: {len(df_all)}")
print(f"  - Qualitative: {len(df_qualitative)}")
print(f"  - Quantitative: {len(df_quantitative)}")
df_all

## 5. Create Annotation Template

Add columns for manual annotation:

| Column | Description |
|--------|-------------|
| `citation_exists` | Does the cited document/page exist? (Yes/No/Partial) |
| `citation_correct` | Is the citation correct for the claim? (Yes/No/Partial) |
| `agent_response` | The response generated by the agent |
| `agent_citation` | The citation provided by the agent (document, page) |
| `notes` | Annotator notes |

In [ ]:
# Create annotation template
df_annotation = df_all.copy()

# # Add manual annotation columns
df_annotation["citation_exist_single_agent"]=""
df_annotation["citation_correct_single_agent"]=""
df_annotation["citation_exist_multi_agent"]=""
df_annotation["citation_correct_multi_agent"]=""

# # Add agent output columns (to be filled from agent runs)
# df_annotation["agent_type"] = ""  # single_agent or multi_agent
# df_annotation["agent_response"] = ""
# df_annotation["agent_value"] = ""  # For quantitative KPIs
# df_annotation["agent_citation_document"] = ""
# df_annotation["agent_citation_pages"] = ""


df_annotation.head()

In [ ]:
# Display column info
print("Annotation DataFrame Columns:")
print("="*50)
for col in df_annotation.columns:
    print(f"  - {col}")

## 6. Export Annotation Template to CSV

In [ ]:
# Export empty annotation template
template_path = os.path.join(OUTPUT_DIR, f"{CLIENT}_citation_annotation_template.csv")
df_annotation.to_csv(template_path, index=False)
print(f"Annotation template saved to: {template_path}")

## 7. Load Existing Agent Results (Optional)

If you have existing agent run results, load them here to pre-populate the annotation dataframe.

In [ ]:
# Example: Load existing results from citation folder
# Change file paths as needed
existing_results_path = os.path.join(OUTPUT_DIR, "citation")

if os.path.exists(existing_results_path):
    csv_files = [f for f in os.listdir(existing_results_path) if f.endswith('.csv')]
    print(f"Found {len(csv_files)} CSV files in {existing_results_path}:")
    for f in csv_files:
        print(f"  - {f}")
else:
    print(f"No existing results directory found at: {existing_results_path}")

## 8. Load Annotated Data for Analysis

In [ ]:
# After manual annotation, reload the CSV
annotated_path = os.path.join(OUTPUT_DIR, "citation",f"{CLIENT}_citation_annotation_completed.csv")

if os.path.exists(annotated_path):
    df_annotated = pd.read_csv(annotated_path,  sep=";")
    print(f"Loaded {len(df_annotated)} annotated rows")
    
    # Summary statistics
    print("\n" + "="*50)
    print("Annotation Summary")
    print("="*50)
    
    for col in ["citation_exist_single_agent","citation_correct_single_agent","citation_exist_multi_agent","citation_correct_multi_agent"]:
        if col in df_annotated.columns:
            print(f"\n{col}:")
            print(df_annotated[col].value_counts())
else:
    print(f"No annotated file found at: {annotated_path}")
    print("Complete annotation in the template CSV and save as '_completed.csv'")

## Citation Annotation Statistics

**Annotation Codes:**
- 0 = Missing (no citation provided)
- 1 = Correct (citation is accurate)
- 2 = Correct but too broad (citation includes many pages)
- 3 = Partially correct

In [ ]:
# Define annotation code labels
ANNOTATION_LABELS = {
    0: "Missing",
    1: "Correct",
    2: "Correct (broad)",
    3: "Partial"
}

# Columns to analyze
citation_cols = [
    "citation_exist_single_agent",
    "citation_correct_single_agent",
    "citation_exist_multi_agent",
    "citation_correct_multi_agent"
]

# Summary statistics table
summary_data = []
for col in citation_cols:
    if col in df_annotated.columns:
        counts = df_annotated[col].value_counts().sort_index()
        total = len(df_annotated[col].dropna())
        
        row = {
            "Column": col.replace("_", " ").title(),
            "Total": total,
            "Missing (0)": counts.get(0, 0),
            "Correct (1)": counts.get(1, 0),
            "Broad (2)": counts.get(2, 0),
            "Partial (3)": counts.get(3, 0),
        }
        
        # Calculate percentages
        row["Correct %"] = round((counts.get(1, 0) / total) * 100, 1) if total > 0 else 0
        row["Correct+Broad %"] = round(((counts.get(1, 0) + counts.get(2, 0)) / total) * 100, 1) if total > 0 else 0
        row["Any Valid %"] = round(((counts.get(1, 0) + counts.get(2, 0) + counts.get(3, 0)) / total) * 100, 1) if total > 0 else 0
        
        summary_data.append(row)

df_summary = pd.DataFrame(summary_data)
print("Citation Annotation Summary")
print("=" * 80)
df_summary

In [ ]:
# Comparison: Single-Agent vs Multi-Agent
print("\n" + "=" * 60)
print("Single-Agent vs Multi-Agent Comparison")
print("=" * 60)

comparison_metrics = []

for metric_type in ["exist", "correct"]:
    single_col = f"citation_{metric_type}_single_agent"
    multi_col = f"citation_{metric_type}_multi_agent"
    
    if single_col in df_annotated.columns and multi_col in df_annotated.columns:
        single_vals = df_annotated[single_col].dropna()
        multi_vals = df_annotated[multi_col].dropna()
        
        # Strict correct (only 1)
        single_correct = (single_vals == 1).sum()
        multi_correct = (multi_vals == 1).sum()
        
        # Lenient correct (1 or 2)
        single_lenient = ((single_vals == 1) | (single_vals == 2)).sum()
        multi_lenient = ((multi_vals == 1) | (multi_vals == 2)).sum()
        
        # Any valid (1, 2, or 3)
        single_any = ((single_vals == 1) | (single_vals == 2) | (single_vals == 3)).sum()
        multi_any = ((multi_vals == 1) | (multi_vals == 2) | (multi_vals == 3)).sum()
        
        n = len(single_vals)
        
        comparison_metrics.append({
            "Metric": f"Citation {metric_type.title()}",
            "Single Correct": f"{single_correct}/{n} ({single_correct/n*100:.1f}%)",
            "Multi Correct": f"{multi_correct}/{n} ({multi_correct/n*100:.1f}%)",
            "Single Lenient": f"{single_lenient}/{n} ({single_lenient/n*100:.1f}%)",
            "Multi Lenient": f"{multi_lenient}/{n} ({multi_lenient/n*100:.1f}%)",
        })

df_comparison = pd.DataFrame(comparison_metrics)
df_comparison

## Visualization: Citation Accuracy Distribution

In [ ]:
import matplotlib.pyplot as plt

# Color palette for annotation codes
colors = {
    0: "#d62728",  # Red - Missing
    1: "#2ca02c",  # Green - Correct
    2: "#1f77b4",  # Blue - Correct but broad
    3: "#ff7f0e",  # Orange - Partial
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, col in enumerate(citation_cols):
    if col in df_annotated.columns:
        ax = axes[idx]
        counts = df_annotated[col].value_counts().sort_index()
        
        # Create bar chart
        bars = ax.bar(
            [ANNOTATION_LABELS.get(k, str(k)) for k in counts.index],
            counts.values,
            color=[colors.get(k, "#999999") for k in counts.index]
        )
        
        # Add value labels on bars
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                   str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
        
        ax.set_title(col.replace("_", " ").title(), fontsize=12, fontweight='bold')
        ax.set_ylabel("Count")
        ax.set_xlabel("Annotation")

plt.suptitle(f"Citation Annotation Distribution - {CLIENT}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar chart: Single vs Multi-Agent comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, metric_type in enumerate(["exist", "correct"]):
    ax = axes[idx]
    single_col = f"citation_{metric_type}_single_agent"
    multi_col = f"citation_{metric_type}_multi_agent"
    
    if single_col in df_annotated.columns and multi_col in df_annotated.columns:
        # Get counts for each code
        single_counts = df_annotated[single_col].value_counts().sort_index()
        multi_counts = df_annotated[multi_col].value_counts().sort_index()
        
        # Ensure all codes are present
        all_codes = sorted(set(single_counts.index) | set(multi_counts.index))
        single_vals = [single_counts.get(c, 0) for c in all_codes]
        multi_vals = [multi_counts.get(c, 0) for c in all_codes]
        
        x = np.arange(len(all_codes))
        width = 0.35
        
        bars1 = ax.bar(x - width/2, single_vals, width, label='Single-Agent', color='#1f77b4')
        bars2 = ax.bar(x + width/2, multi_vals, width, label='Multi-Agent', color='#ff7f0e')
        
        # Add value labels
        for bar in bars1:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                   str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
        for bar in bars2:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                   str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
        
        ax.set_xlabel('Annotation Code')
        ax.set_ylabel('Count')
        ax.set_title(f'Citation {metric_type.title()}: Single vs Multi-Agent', fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels([ANNOTATION_LABELS.get(c, str(c)) for c in all_codes])
        ax.legend()

plt.suptitle(f"Citation Accuracy Comparison - {CLIENT}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Stacked percentage bar chart
fig, ax = plt.subplots(figsize=(10, 6))

# Prepare data for stacked bars
categories = []
data_by_code = {0: [], 1: [], 2: [], 3: []}

for col in citation_cols:
    if col in df_annotated.columns:
        counts = df_annotated[col].value_counts()
        total = counts.sum()
        
        # Clean up column name for display
        display_name = col.replace("citation_", "").replace("_", " ").title()
        categories.append(display_name)
        
        for code in [0, 1, 2, 3]:
            pct = (counts.get(code, 0) / total) * 100 if total > 0 else 0
            data_by_code[code].append(pct)

x = np.arange(len(categories))
width = 0.6

# Stack the bars
bottom = np.zeros(len(categories))
bar_colors = {0: "#d62728", 1: "#2ca02c", 2: "#1f77b4", 3: "#ff7f0e"}

for code in [0, 1, 2, 3]:
    bars = ax.bar(x, data_by_code[code], width, bottom=bottom, 
                  label=ANNOTATION_LABELS[code], color=bar_colors[code])
    
    # Add percentage labels on each segment (if > 5%)
    for i, (val, b) in enumerate(zip(data_by_code[code], bottom)):
        if val > 5:
            ax.text(i, b + val/2, f'{val:.0f}%', ha='center', va='center', 
                   fontsize=9, fontweight='bold', color='white')
    
    bottom += np.array(data_by_code[code])

ax.set_ylabel('Percentage (%)')
ax.set_title(f'Citation Accuracy Distribution (Stacked) - {CLIENT}', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=15, ha='right')
ax.legend(loc='upper right')
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

## Breakdown by KPI Type (Qualitative vs Quantitative)

In [ ]:
# Check if kpi_type column exists
if "kpi_type" in df_annotated.columns:
    print("Citation Accuracy by KPI Type")
    print("=" * 70)
    
    for kpi_type in ["qualitative", "quantitative"]:
        df_type = df_annotated[df_annotated["kpi_type"] == kpi_type]
        print(f"\n{kpi_type.upper()} (n={len(df_type)})")
        print("-" * 50)
        
        for col in citation_cols:
            if col in df_type.columns:
                counts = df_type[col].value_counts()
                total = len(df_type)
                correct = counts.get(1, 0)
                correct_broad = counts.get(1, 0) + counts.get(2, 0)
                
                print(f"  {col.replace('citation_', '').replace('_', ' ').title()}: "
                      f"Correct={correct}/{total} ({correct/total*100:.1f}%), "
                      f"Lenient={correct_broad}/{total} ({correct_broad/total*100:.1f}%)")
else:
    print("kpi_type column not found in annotated data")

## Summary Statistics Table

In [ ]:
# Final summary table for thesis - separated by KPI type
print("=" * 80)
print("CITATION ACCURACY SUMMARY FOR THESIS")
print("=" * 80)

summary_thesis = []

# Define KPI types to analyze
kpi_types = ["All", "qualitative", "quantitative"]

for kpi_type in kpi_types:
    # Filter data by KPI type
    if kpi_type == "All":
        df_filtered = df_annotated
    else:
        df_filtered = df_annotated[df_annotated["kpi_type"] == kpi_type]
    
    n = len(df_filtered)
    if n == 0:
        continue
    
    for agent in ["single_agent", "multi_agent"]:
        exist_col = f"citation_exist_{agent}"
        correct_col = f"citation_correct_{agent}"
        
        if exist_col in df_filtered.columns and correct_col in df_filtered.columns:
            # Citation Exists
            exist_vals = df_filtered[exist_col]
            exist_correct = (exist_vals == 1).sum()
            exist_broad = ((exist_vals == 1) | (exist_vals == 2)).sum()
            
            # Citation Correct
            correct_vals = df_filtered[correct_col]
            correct_correct = (correct_vals == 1).sum()
            correct_broad = ((correct_vals == 1) | (correct_vals == 2)).sum()
            
            summary_thesis.append({
                "KPI Type": kpi_type.title(),
                "Agent": agent.replace("_", " ").title(),
                "N": n,
                "Citation Exists (Strict)": f"{exist_correct/n*100:.1f}%",
                "Citation Exists (Lenient)": f"{exist_broad/n*100:.1f}%",
                "Citation Correct (Strict)": f"{correct_correct/n*100:.1f}%",
                "Citation Correct (Lenient)": f"{correct_broad/n*100:.1f}%",
            })

df_thesis_summary = pd.DataFrame(summary_thesis)
print("\nSummary Table:")
print(df_thesis_summary.to_string(index=False))

# Also display as formatted dataframe
df_thesis_summary

In [ ]:
# Visualization: Separated by Qualitative vs Quantitative
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, kpi_type in enumerate(["qualitative", "quantitative"]):
    ax = axes[idx]
    df_type = df_annotated[df_annotated["kpi_type"] == kpi_type]
    n = len(df_type)
    
    # Prepare data for grouped bars
    metrics = []
    single_strict = []
    single_lenient = []
    multi_strict = []
    multi_lenient = []
    
    for metric in ["exist", "correct"]:
        single_col = f"citation_{metric}_single_agent"
        multi_col = f"citation_{metric}_multi_agent"
        
        if single_col in df_type.columns and multi_col in df_type.columns:
            metrics.append(f"Citation\n{metric.title()}")
            
            single_strict.append((df_type[single_col] == 1).sum() / n * 100)
            single_lenient.append(((df_type[single_col] == 1) | (df_type[single_col] == 2)).sum() / n * 100)
            multi_strict.append((df_type[multi_col] == 1).sum() / n * 100)
            multi_lenient.append(((df_type[multi_col] == 1) | (df_type[multi_col] == 2)).sum() / n * 100)
    
    x = np.arange(len(metrics))
    width = 0.2
    
    bars1 = ax.bar(x - 1.5*width, single_strict, width, label='Single (Strict)', color='#1f77b4')
    bars2 = ax.bar(x - 0.5*width, single_lenient, width, label='Single (Lenient)', color='#aec7e8')
    bars3 = ax.bar(x + 0.5*width, multi_strict, width, label='Multi (Strict)', color='#ff7f0e')
    bars4 = ax.bar(x + 1.5*width, multi_lenient, width, label='Multi (Lenient)', color='#ffbb78')
    
    # Add value labels
    for bars in [bars1, bars2, bars3, bars4]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, height + 1,
                   f'{height:.0f}%', ha='center', va='bottom', fontsize=9)
    
    ax.set_ylabel('Accuracy (%)')
    ax.set_title(f'{kpi_type.title()} KPIs (n={n})', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(0, 110)

plt.suptitle(f"Citation Accuracy: Qualitative vs Quantitative - {CLIENT}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()